# Dataset507 Results Preview -- baseline vs. two-channel refinement

Compares `nnUNetTrainerSmallENet` (baseline, image-only) against
`nnUNetTrainerSmallRefinementENet` (image + first-pass predicted mask,
`analysis/507_refinement_net_plan.md`) on `Dataset507_ARCADE_refinement`'s
held-out test split.

**Why this notebook exists, specifically**: both models finished 150
epochs with a large gap in pooled pseudo-dice (baseline ~0.79, refinement
~0.88), but the refinement net's training curve is suspicious -- it started
at ~0.87 pseudo-dice at epoch 0 and barely moved from there. Since
Dataset507 is 75% patches (`normal` + `empty_fp`) where the first-pass
prediction (the network's *second input channel*) is already correct or
already empty, a network can reach a high pooled dice just by learning to
mostly copy that channel through, without doing anything useful with the
image on the 25% `discont` patches -- the actual reason this dataset
exists. A pooled number can't tell those two stories apart; a per-category
breakdown can. That's what this notebook is for.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import torch
from PIL import Image

# Checkpoints were pickled under numpy>=2.0 on the HPC training environment;
# this container has numpy 1.26.4 (required for torch 2.0.1's numpy interop --
# numpy>=2.0 breaks torch.from_numpy entirely here). numpy 1.26.4 ships a
# forward-compat numpy.dtypes.Float64DType class but its own
# numpy.core.multiarray.scalar() unpickling function doesn't accept instances
# of it, so torch.load(..., weights_only=False) fails on any tensor whose
# saved dtype used the 2.x class path. Patch scalar() to coerce it back to a
# real np.dtype first.
import numpy.core.multiarray as _ma

_orig_scalar = _ma.scalar


def _patched_scalar(dtype, obj):
    if not isinstance(dtype, np.dtype):
        try:
            dtype = np.dtype(dtype.type) if hasattr(dtype, "type") else np.dtype(dtype)
        except Exception as e:
            raise TypeError(f"could not coerce {dtype!r} to np.dtype") from e
    return _orig_scalar(dtype, obj)


_ma.scalar = _patched_scalar


def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "lightm-unet").exists() and (candidate / "data").exists():
            return candidate
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
PACKAGE_ROOT = REPO_ROOT / "lightm-unet"
sys.path.insert(0, str(PACKAGE_ROOT))
from nnunetv2.nets.SmallENet import SmallENet  # noqa: E402
from nnunetv2.nets.SmallRefinementENet import SmallRefinementENet  # noqa: E402

DATASET_NAME = "Dataset507_ARCADE_refinement"
PLANS_NAME = "nnUNetPlans"
CONFIGURATION = "2d"
FOLD = 0
CHECKPOINT_NAME = "checkpoint_best.pth"

DATASET_DIR = REPO_ROOT / "data" / "nnUNet_raw" / DATASET_NAME
IMAGES_TS_DIR = DATASET_DIR / "imagesTs"
LABELS_TS_DIR = DATASET_DIR / "labelsTs"
NNUNET_RESULTS = REPO_ROOT / "data" / "nnUNET_results"

BASELINE_TRAINER = "nnUNetTrainerSmallENet"
REFINEMENT_TRAINER = "nnUNetTrainerSmallRefinementENet"

BASELINE_CKPT = NNUNET_RESULTS / DATASET_NAME / f"{BASELINE_TRAINER}__{PLANS_NAME}__{CONFIGURATION}" / f"fold_{FOLD}" / CHECKPOINT_NAME
REFINEMENT_CKPT = NNUNET_RESULTS / DATASET_NAME / f"{REFINEMENT_TRAINER}__{PLANS_NAME}__{CONFIGURATION}" / f"fold_{FOLD}" / CHECKPOINT_NAME

METRICS_DIR = REPO_ROOT / "analysis" / "507_ARCADE_refinement" / "results"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

# Must match nnUNetTrainerSmallENet.py / SmallRefinementENet defaults used at training time.
INITIAL_CHANNELS = 16
STAGE_CHANNELS = 32
LCN_KERNEL_SIZE = 9
STEM_CHANNELS = 8
THRESHOLD = 0.5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Repo root        :", REPO_ROOT)
print("Dataset dir      :", DATASET_DIR)
print("Baseline ckpt    :", BASELINE_CKPT, BASELINE_CKPT.exists())
print("Refinement ckpt  :", REFINEMENT_CKPT, REFINEMENT_CKPT.exists())
print("Device           :", DEVICE)

## Load both models

In [ ]:
def load_checkpoint_into(model: torch.nn.Module, path: Path) -> dict:
    checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["network_weights"])
    return checkpoint


baseline_model = SmallENet(
    in_channels=1, out_channels=1,
    initial_channels=INITIAL_CHANNELS, stage_channels=STAGE_CHANNELS, lcn_kernel_size=LCN_KERNEL_SIZE,
)
baseline_checkpoint = load_checkpoint_into(baseline_model, BASELINE_CKPT)
baseline_model = baseline_model.to(DEVICE).eval()

refinement_model = SmallRefinementENet(stem_channels=STEM_CHANNELS, stage_channels=STAGE_CHANNELS)
refinement_checkpoint = load_checkpoint_into(refinement_model, REFINEMENT_CKPT)
refinement_model = refinement_model.to(DEVICE).eval()

print(f"Baseline: epoch {baseline_checkpoint.get('current_epoch')}, "
      f"{sum(p.numel() for p in baseline_model.parameters()):,} params")
print(f"Refinement: epoch {refinement_checkpoint.get('current_epoch')}, "
      f"{sum(p.numel() for p in refinement_model.parameters()):,} params")

## Sample index -- parse category from case id

Case ids are `{source_stem}_{category}_{idx:03d}` (`category` in `discont` /
`normal` / `empty`), per `dataset-prep/README_507_refinement.md`.

In [ ]:
def case_id_from_image(path: Path) -> str:
    stem = path.stem
    return stem[:-5] if stem.endswith("_0000") else stem


def category_of(case_id: str) -> str:
    return case_id.rsplit("_", 2)[1]


img_files = sorted(IMAGES_TS_DIR.glob("*_0000.png"))
gt_files = {p.stem: p for p in LABELS_TS_DIR.glob("*.png")}

samples = []
for img in img_files:
    cid = case_id_from_image(img)
    if cid in gt_files:
        samples.append({
            "case_id": cid, "category": category_of(cid),
            "image": img, "pred_mask": IMAGES_TS_DIR / f"{cid}_0001.png", "gt": gt_files[cid],
        })

print(f"Matched test samples: {len(samples)}")
cat_counts = pd.Series([s["category"] for s in samples]).value_counts()
print(cat_counts)

## Inference helpers

Both channels use `ZScoreNormalization` independently (confirmed from
Dataset507's `nnUNetPlans.json`: `normalization_schemes: ['ZScoreNormalization',
'ZScoreNormalization']`, `use_mask_for_norm: [False, False]`) -- each
channel is normalized by its own mean/std, not jointly.

In [ ]:
def load_gray(path: Path) -> np.ndarray:
    return np.asarray(Image.open(path).convert("L"), dtype=np.float32)


def load_gt(path: Path) -> np.ndarray:
    arr = np.asarray(Image.open(path))
    if arr.ndim == 3:
        arr = arr[..., 0]
    return arr


def zscore_normalize(image: np.ndarray) -> np.ndarray:
    mean = image.mean()
    std = image.std()
    return (image - mean) / max(std, 1e-8)


# Both networks were trained on 96x96 inputs -- nnU-Net's plans set
# patch_size=[96, 96] for this dataset (confirmed in nnUNetPlans.json), but
# the *preprocessed* .npy patches stay at their native 85x85 (confirmed:
# no spacing resampling happens since spacing is already 1.0). The 85->96
# gap is closed on-the-fly by nnUNetDataLoader2D.get_bbox, which -- for a
# patch this much smaller than patch_size -- deterministically zero-pads
# (in already-normalized space, `np.pad(..., 'constant', constant_values=0)`)
# 5px before / 6px after on each axis (need_to_pad=11, split via
# `-need_to_pad//2` / `need_to_pad//2 + need_to_pad%2`; see
# base_data_loader.py's get_bbox). SmallRefinementENet's down/up round-trip
# also requires even H/W, so raw 85x85 inference fails outright
# (RuntimeError from UpsamplingBottleneck) -- this padding is not optional
# for it, and matching it for the baseline too keeps both models evaluated
# under the same input distribution they were trained on.
PATCH_SIZE = 96
_native = 85
_need = PATCH_SIZE - _native
PAD_BEFORE = _need // 2       # 5
PAD_AFTER = _need - PAD_BEFORE  # 6


def pad_to_patch(arr: np.ndarray) -> np.ndarray:
    return np.pad(arr, ((PAD_BEFORE, PAD_AFTER), (PAD_BEFORE, PAD_AFTER)), mode="constant", constant_values=0)


def crop_from_patch(arr: np.ndarray) -> np.ndarray:
    return arr[PAD_BEFORE:PAD_BEFORE + _native, PAD_BEFORE:PAD_BEFORE + _native]


@torch.no_grad()
def predict_case(idx: int) -> dict:
    item = samples[idx]
    raw = load_gray(item["image"])
    pred_mask_raw = load_gray(item["pred_mask"])
    gt = load_gt(item["gt"]) > 0

    raw_n = pad_to_patch(zscore_normalize(raw))
    pred_mask_n = pad_to_patch(zscore_normalize(pred_mask_raw))

    x1 = torch.from_numpy(raw_n).float()[None, None].to(DEVICE)  # 1,1,96,96
    baseline_logits = baseline_model(x1)
    baseline_prob = crop_from_patch(torch.sigmoid(baseline_logits)[0, 0].cpu().numpy())

    x2 = torch.stack([torch.from_numpy(raw_n).float(), torch.from_numpy(pred_mask_n).float()], dim=0)[None].to(DEVICE)  # 1,2,96,96
    refinement_logits = refinement_model(x2)
    refinement_prob = crop_from_patch(torch.sigmoid(refinement_logits)[0, 0].cpu().numpy())

    return {
        "raw": raw, "pred_mask_input": pred_mask_raw > 127, "gt": gt,
        "baseline_prob": baseline_prob, "refinement_prob": refinement_prob,
    }

## Metrics over the full test split, per category

The key table. Pooled dice matches what the training log implied
(refinement >> baseline overall) -- the question is whether that holds up
*within* the `discont` category specifically, or whether it's driven by
`normal`/`empty` where copying the input predicted mask already wins.

In [ ]:
EPS = 1e-8


def prf_from_counts(tp: int, fp: int, fn: int, tn: int) -> dict[str, float]:
    precision = tp / (tp + fp + EPS)
    recall = tp / (tp + fn + EPS)
    dice = (2 * tp) / (2 * tp + fp + fn + EPS)
    accuracy = (tp + tn) / (tp + fp + fn + tn + EPS)
    return {"dice_f1": dice, "precision": precision, "recall_sensitivity": recall, "accuracy": accuracy}


def counts(gt: np.ndarray, pred: np.ndarray) -> tuple[int, int, int, int]:
    tp = int((gt & pred).sum()); fp = int((~gt & pred).sum())
    fn = int((gt & ~pred).sum()); tn = int((~gt & ~pred).sum())
    return tp, fp, fn, tn


rows = []
for idx, item in enumerate(samples):
    out = predict_case(idx)
    gt = out["gt"]
    baseline_pred = out["baseline_prob"] > THRESHOLD
    refinement_pred = out["refinement_prob"] > THRESHOLD

    row = {"case_id": item["case_id"], "category": item["category"], "gt_vessel_px": int(gt.sum())}
    row.update({f"baseline_{k}": v for k, v in prf_from_counts(*counts(gt, baseline_pred)).items()})
    row.update({f"refinement_{k}": v for k, v in prf_from_counts(*counts(gt, refinement_pred)).items()})
    rows.append(row)

per_image_metrics = pd.DataFrame(rows)
per_image_metrics.to_csv(METRICS_DIR / "per_image_metrics.csv", index=False)

print("Overall (pooled mean across all test patches):")
display(per_image_metrics[["baseline_dice_f1", "refinement_dice_f1"]].mean().to_frame("mean_dice"))

print("\nPer-category mean dice -- THE KEY RESULT:")
category_summary = (
    per_image_metrics.groupby("category")[["baseline_dice_f1", "refinement_dice_f1"]]
    .mean()
    .assign(refinement_minus_baseline=lambda d: d["refinement_dice_f1"] - d["baseline_dice_f1"])
    .reindex(["discont", "normal", "empty"])
)
display(category_summary)
category_summary.to_csv(METRICS_DIR / "category_summary.csv")
print("Saved to", METRICS_DIR)

## Visual comparison, a few examples per category

Raw / input predicted-mask channel / GT / baseline prediction / refinement
prediction, for a handful of cases from each category.

In [ ]:
IMSHOW_MASK_KW = dict(cmap=mcolors.ListedColormap(["#000000", "#ffffff"]), norm=mcolors.BoundaryNorm([0, 1, 2], 2), interpolation="nearest")

rng = np.random.default_rng(0)


def show_category(category: str, n: int = 4):
    indices = [i for i, s in enumerate(samples) if s["category"] == category]
    chosen = rng.choice(indices, size=min(n, len(indices)), replace=False)
    fig, axes = plt.subplots(len(chosen), 5, figsize=(13, 2.8 * len(chosen)))
    axes = np.atleast_2d(axes)
    for col, title in enumerate(["Raw", "Input pred mask (ch. 1)", "GT", "Baseline pred", "Refinement pred"]):
        axes[0, col].set_title(title, fontsize=9)
    for row, idx in enumerate(chosen):
        out = predict_case(idx)
        b_dice = per_image_metrics.loc[idx, "baseline_dice_f1"]
        r_dice = per_image_metrics.loc[idx, "refinement_dice_f1"]
        axes[row, 0].imshow(out["raw"], cmap="gray")
        axes[row, 0].set_ylabel(f"{samples[idx]['case_id']}\nb={b_dice:.2f} r={r_dice:.2f}", fontsize=6.5, rotation=0, labelpad=48, va="center")
        axes[row, 0].axis("off")
        axes[row, 1].imshow(out["pred_mask_input"], **IMSHOW_MASK_KW); axes[row, 1].axis("off")
        axes[row, 2].imshow(out["gt"], **IMSHOW_MASK_KW); axes[row, 2].axis("off")
        axes[row, 3].imshow(out["baseline_prob"] > THRESHOLD, **IMSHOW_MASK_KW); axes[row, 3].axis("off")
        axes[row, 4].imshow(out["refinement_prob"] > THRESHOLD, **IMSHOW_MASK_KW); axes[row, 4].axis("off")
    fig.suptitle(f"category: {category}", fontweight="bold")
    plt.tight_layout()
    fig.savefig(METRICS_DIR / f"category_{category}_examples.png", dpi=150, bbox_inches="tight")
    plt.show()


for cat in ["discont", "normal", "empty"]:
    show_category(cat)

## Conclusion

Fill in after running: does `refinement_minus_baseline` on the `discont`
row hold up, or is it near zero while `normal`/`empty` carry the pooled
average? That answers whether the two-channel network learned genuine
gap-filling or a copy-the-input-mask shortcut.